# Differece calculations between frames

In [1]:
import numpy as np 
import pandas as pd 
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt 
import cv2

## Load in the data

In [ ]:
def get_data(dataset,X,num=-1,augment=False,sobel=True,printer="resin",filament="resin",pressure="P30",pattern="",augtests=[]):
    subset = dataset[((dataset['Printer'] == printer) & (dataset['Filament']==filament)) ]
    if pattern!="":
        subset = subset[subset['Pattern'].str.contains(pattern, regex=False, na=False) ]
    subX, y = [], []
    for _, row in subset.iterrows():
        images=X[int(row['Index'])]
        images=images.reshape((2*25,images.shape[2],images.shape[3]))
        try:
            label=int(row['Pattern'].split(".")[0].replace("z", ""))
        except:
            label=""
        if label!="":
            for image in images:
                if sobel:
                    # Apply Sobel filter (on grayscale if not already)
                    if len(image.shape) == 3:  # convert to grayscale if it's RGB
                        roi_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                    else:
                        roi_gray = image

                    sobelx = cv2.Sobel(roi_gray, cv2.CV_64F, 1, 0, ksize=3)
                    sobely = cv2.Sobel(roi_gray, cv2.CV_64F, 0, 1, ksize=3)

                    sobel_ = np.hypot(sobelx, sobely)  # magnitude
                    image = cv2.convertScaleAbs(sobel_)  # scale back to uint8
                if num==-1:
                    y.append(label)
                    subX.append(image)
                else: 
                    try:
                        counter=int(row['Pattern'].split(".")[1])
                        if counter==num:
                            y.append(label)
                            subX.append(image)
                    except:
                        pass
    subX,y=np.array(subX),np.array(y)
    if augment:
        if len(augtests)==0 or 0 in augtests:
            noisy=subX+np.random.normal(0,5,subX.shape)
            subX=np.concatenate([subX,noisy])
            y=np.concatenate([y,y])
        if len(augtests)==1 or 1 in augtests:
            light=subX.copy()-10
            dark=subX.copy()+10
            light[light<0]=0
            dark[dark>255]=255
            subX=np.concatenate([subX,light,dark])
            y=np.concatenate([y,y,y])
    assert len(subX)==len(y) 
    return subX.reshape((len(subX)//(2*25),2,25,subX.shape[1],subX.shape[2])),y #shape into correct format

def coral_align_target_to_source(Xs, Xt, eps=1e-6):
    # center
    mu_s = Xs.mean(axis=0, keepdims=True)
    mu_t = Xt.mean(axis=0, keepdims=True)
    Xs_c = Xs - mu_s
    Xt_c = Xt - mu_t

    # covariances with small ridge
    cov_s = np.cov(Xs_c, rowvar=False) + np.eye(Xs_c.shape[1]) * eps
    cov_t = np.cov(Xt_c, rowvar=False) + np.eye(Xt_c.shape[1]) * eps

    # matrix square-roots via SVD
    Us, Ss, _ = np.linalg.svd(cov_s)
    Ut, St, _ = np.linalg.svd(cov_t)
    # cov_s^{1/2} and cov_t^{-1/2}
    cov_s_sqrt = Us @ np.diag(np.sqrt(Ss)) @ Us.T
    cov_t_inv_sqrt = Ut @ np.diag(1.0/np.sqrt(St)) @ Ut.T

    # transform target: Xt_aligned = (Xt_c @ cov_t^{-1/2}) @ cov_s^{1/2} + mu_s
    Xt_aligned = (Xt_c @ cov_t_inv_sqrt) @ cov_s_sqrt + mu_s
    return Xt_aligned

X=np.load("/home/dexter/Documents/GitHub/3D-textures/Experimental/data/X.npy")
dataset=pd.read_csv("/home/dexter/Documents/GitHub/3D-textures/Experimental/datameta.csv")
print("Dataset size:",X.shape)
print(len(dataset['Filament'].unique()),len(dataset['Printer'].unique()))
printers=[["resin","resin"],["ender","PLAplus"],["bambu","PLAminus"],]


Dataset size: (204, 2, 25, 192, 256)
5 7


In [9]:
dataset.head()

,Unnamed: 0,Index,Filament,Pattern,Printer,Pressure
0,0,0.0,PLAminus,z1.2,bambu,P30
1,1,1.0,PLAminus,z4.1,bambu,P30
2,2,2.0,wood,z0,bambu,P30
3,3,3.0,wood,z3,bambu,P30
4,4,4.0,resin,z5.5,resin,P30


## calculate

In [14]:
for j in range(len(printers)):
    comb = printers[j]
    print("###################################")
    for i in range(0, 6):
        X_c, y_c = get_data(
            dataset, X,
            printer=comb[0],
            filament=comb[1],
            pattern=str(i)
        )

        print(X_c.shape)

        diff = (X_c[:, 0] - X_c[:, 1]) ** 2

        # One MSR value for each of the 10 samples
        msr = np.mean(diff, axis=(1, 2, 3))

        # Std across the 10 samples
        msr_std = np.std(msr)

        print(
            "Texture:", i,
            ", Printer:", comb[0],
            "MSR:", np.mean(msr),
            "±", msr_std
        )

###################################
(5, 2, 25, 192, 256)
Texture: 0 , Printer: resin MSR: 19.4630439453125 ± 5.6468432763242475
(10, 2, 25, 192, 256)
Texture: 1 , Printer: resin MSR: 18.39285799153646 ± 4.510303896453516
(10, 2, 25, 192, 256)
Texture: 2 , Printer: resin MSR: 16.4996025390625 ± 0.5638648675072193
(10, 2, 25, 192, 256)
Texture: 3 , Printer: resin MSR: 16.938008870442708 ± 1.796713159747803
(10, 2, 25, 192, 256)
Texture: 4 , Printer: resin MSR: 16.365294189453124 ± 0.6797474960720529
(10, 2, 25, 192, 256)
Texture: 5 , Printer: resin MSR: 16.499925048828125 ± 0.5426407762516133
###################################
(6, 2, 25, 192, 256)
Texture: 0 , Printer: ender MSR: 18.851731906467013 ± 6.944658924032226
(11, 2, 25, 192, 256)
Texture: 1 , Printer: ender MSR: 17.582102864583334 ± 4.003713663018978
(11, 2, 25, 192, 256)
Texture: 2 , Printer: ender MSR: 17.391021025686552 ± 3.232778118856227
(11, 2, 25, 192, 256)
Texture: 3 , Printer: ender MSR: 17.44950927734375 ± 3.21819582